In [1]:
'''
The sentence_transformers library does not have a built-in Python function (like list_models()) to return a local array of every available model. Instead, you can programmatically list all community and official models by querying the Hugging Face Hub API using the huggingface_hub client, or by looking at the official documentation.
'''
from huggingface_hub import HfApi

# Initialize the Hugging Face API client
api = HfApi()

# Filter models by the 'sentence-transformers' library tag
models = api.list_models(filter="sentence-transformers")

# Print the first 20 model IDs
for i, model in enumerate(models):
    if i >= 20:
        break
    print(model.modelId)

BAAI/bge-m3
sentence-transformers/all-MiniLM-L6-v2
google/embeddinggemma-300m
Qwen/Qwen3-Embedding-0.6B
Qwen/Qwen3-Embedding-8B
LiquidAI/LFM2.5-ColBERT-350M
BAAI/bge-reranker-v2-m3
sionic-ai/comsat-embed-ko-8b-preview
microsoft/harrier-oss-v1-0.6b
sentence-transformers/all-mpnet-base-v2
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
intfloat/multilingual-e5-small
intfloat/multilingual-e5-large
nomic-ai/nomic-embed-text-v2-moe
cointegrated/rubert-tiny2
BAAI/bge-small-en-v1.5
jinaai/jina-clip-v2
Qwen/Qwen3-Reranker-8B
microsoft/harrier-oss-v1-270m
jinaai/jina-embeddings-v5-omni-small


In [2]:
from sentence_transformers import SentenceTransformer

# smallest model from sentence_transformers, 384 dimensions (initial load: 1m)
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1) #384 dim vectors

d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [6]:
print(v1.dot(dv))
print(v2.dot(dv))

0.3233239
0.019730505


In [5]:
# The notebook will need couple module files from code directory.
import os,sys
module_path = os.path.abspath(os.path.join('..', 'code'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [6]:
from ingest import load_faq_data
documents=load_faq_data()

In [7]:
texts = []
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [8]:
from tqdm.auto import tqdm
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1368

In [9]:
import numpy as np
X = np.array(vectors)
X.shape

(1368, 384)

In [10]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)
scores = X.dot(v_query)

In [11]:
idx = np.argmax(scores)
print(idx, scores[idx])
print(documents[idx])

2 0.7629409
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}


In [12]:
top5 = np.argsort(-scores)[:5]
#top5 = top5[::-1]
scores[top5]

array([0.7629409 , 0.757937  , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [13]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629409
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.757937
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [14]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [15]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(
    query_vector, 
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [16]:
results[1]

{'id': '69d122f12e',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'}

In [17]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [18]:
from rag_helper import RAGVector
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)



In [19]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join. You can start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project while submissions are still being accepted.'